In [2]:
import json
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================
# SETUP DIRECTORY PATHS & DUCKDB CONNECTION
# ==========================================
BASE_DIR = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
RESULT_DIR = BASE_DIR / "results"
FIG_DIR = RESULT_DIR / "figures"
ANALYSIS_DIR = BASE_DIR / "analysis"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = RESULT_DIR / "step03_01_star_schema.duckdb"
assert DB_PATH.exists(), f"Warehouse database not found at {DB_PATH}"

con = duckdb.connect(str(DB_PATH))

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

FINDINGS = {}

# Verify analytical views from Step 3.2 are active
obs_count = int(q("SELECT COUNT(*) AS n FROM vw_olap_observed")["n"][0])
print("=== STEP 3.3: MULTIDIMENSIONAL OLAP CUBE ANALYSIS ===")
print(f"Connected to Warehouse : {DB_PATH.as_posix()}")
print(f"Default View           : vw_olap_observed (is_imputed = FALSE)")
print(f"Observed Fact Rows     : {obs_count:,}")

Matplotlib is building the font cache; this may take a moment.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== STEP 3.3: MULTIDIMENSIONAL OLAP CUBE ANALYSIS ===
Connected to Warehouse : D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/step03_01_star_schema.duckdb
Default View           : vw_olap_observed (is_imputed = FALSE)
Observed Fact Rows     : 35,582,318


In [3]:
# ==========================================
# CUBE 1: TRAIN-CATEGORY OLAP ANALYSIS (ROLL-UP & DRILL-DOWN)
# ==========================================
print("\n=== CUBE 1A: TRAIN CATEGORY PERFORMANCE (vw_olap_observed) ===")

train_cat_df = q("""
SELECT 
    primary_type_code,
    COUNT(*) AS observed_stops,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS traffic_share_pct,
    COUNT(DISTINCT train_no) AS distinct_trains,
    COUNT(DISTINCT station_name) AS distinct_stations,
    ROUND(AVG(delay), 2) AS mean_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_delay_min,
    ROUND(STDDEV(delay), 2) AS std_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 2) AS iqr_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.90), 2) AS p90_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.95), 2) AS p95_delay_min,
    ROUND(AVG(is_early) * 100.0, 2) AS early_lt_0m_pct,
    ROUND(AVG(is_exact_ontime) * 100.0, 2) AS exact_0m_pct,
    ROUND(AVG(is_early_or_ontime) * 100.0, 2) AS early_or_ontime_le_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_high_risk_only) * 100.0, 4) AS high_risk_2sd_3sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct
FROM vw_olap_observed
GROUP BY primary_type_code
ORDER BY observed_stops DESC;
""")
print(train_cat_df.to_string(index=False))

train_cat_csv = RESULT_DIR / "olap_step03_03_train_category.csv"
train_cat_df.to_csv(train_cat_csv, index=False)

print("\n=== CUBE 1B: DRILL-DOWN BY (primary_type_code x is_premium_special) ===")
train_cat_drill_df = q("""
SELECT 
    primary_type_code,
    is_premium_special,
    COUNT(*) AS observed_stops,
    COUNT(DISTINCT train_no) AS distinct_trains,
    ROUND(AVG(delay), 2) AS mean_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_delay_min,
    ROUND(STDDEV(delay), 2) AS std_delay_min,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct
FROM vw_olap_observed
GROUP BY primary_type_code, is_premium_special
ORDER BY primary_type_code, is_premium_special DESC;
""")
print(train_cat_drill_df.to_string(index=False))

train_cat_drill_csv = RESULT_DIR / "olap_step03_03_train_category_premium_drilldown.csv"
train_cat_drill_df.to_csv(train_cat_drill_csv, index=False)

FINDINGS["cube1_train_category"] = {
    "by_primary_type": train_cat_df.to_dict(orient="records"),
    "drilldown_premium_special": train_cat_drill_df.to_dict(orient="records")
}


=== CUBE 1A: TRAIN CATEGORY PERFORMANCE (vw_olap_observed) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

primary_type_code  observed_stops  traffic_share_pct  distinct_trains  distinct_stations  mean_delay_min  median_delay_min  std_delay_min  iqr_delay_min  p90_delay_min  p95_delay_min  early_lt_0m_pct  exact_0m_pct  early_or_ontime_le_0m_pct  on_time_le_15m_pct  delayed_gt_15m_pct  high_risk_2sd_3sd_pct  critical_gt_3sd_pct  bottleneck_gt_2sd_pct
      PASS-TRAINS        17519157            49.2356             3126               7546           32.41              15.0          55.13           38.0           86.0          125.0             2.08         16.04                      18.12               51.58               48.42                 1.5121               1.0027                 2.5148
       EXP-TRAINS        12605410            35.4261             2071               5048           37.14              18.0          66.90           36.0           87.0          140.0             1.06         11.01                      12.07               46.29               53.71                 1.7033 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

primary_type_code  is_premium_special  observed_stops  distinct_trains  mean_delay_min  median_delay_min  std_delay_min  on_time_le_15m_pct  delayed_gt_15m_pct  bottleneck_gt_2sd_pct  critical_gt_3sd_pct
       EXP-TRAINS                True           67441               28           95.11              32.0         175.03               33.74               66.26                15.1881              11.7184
       EXP-TRAINS               False        12537969             2043           36.83              18.0          65.70               46.36               53.64                 3.6780               1.9841
       GRB-TRAINS               False           91939               52           51.44              17.0         109.24               48.21               51.79                 7.5256               5.2165
      PASS-TRAINS                True           64411               17           13.30               4.0          29.97               78.30               21.70                 0.6257  

In [4]:
# ==========================================
# CUBE 2: RAILWAY-ZONE PERFORMANCE ANALYSIS
# ==========================================
print("\n=== CUBE 2: RAILWAY-ZONE PERFORMANCE (ORDERED BY BOTTLENECK RATE) ===")

zone_df = q("""
SELECT 
    station_zone,
    COUNT(*) AS observed_stops,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 4) AS traffic_share_pct,
    COUNT(DISTINCT station_name) AS distinct_stations,
    COUNT(DISTINCT train_no) AS distinct_trains,
    ROUND(AVG(delay), 2) AS mean_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_delay_min,
    ROUND(STDDEV(delay), 2) AS std_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 2) AS iqr_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.90), 2) AS p90_delay_min,
    ROUND(AVG(is_early_or_ontime) * 100.0, 2) AS early_or_ontime_le_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_high_risk_only) * 100.0, 4) AS high_risk_2sd_3sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct
FROM vw_olap_observed
GROUP BY station_zone
ORDER BY bottleneck_gt_2sd_pct DESC;
""")
print(zone_df.to_string(index=False))

zone_csv = RESULT_DIR / "olap_step03_03_railway_zone.csv"
zone_df.to_csv(zone_csv, index=False)

FINDINGS["cube2_railway_zone"] = zone_df.to_dict(orient="records")


=== CUBE 2: RAILWAY-ZONE PERFORMANCE (ORDERED BY BOTTLENECK RATE) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

station_zone  observed_stops  traffic_share_pct  distinct_stations  distinct_trains  mean_delay_min  median_delay_min  std_delay_min  iqr_delay_min  p90_delay_min  early_or_ontime_le_0m_pct  on_time_le_15m_pct  delayed_gt_15m_pct  high_risk_2sd_3sd_pct  critical_gt_3sd_pct  bottleneck_gt_2sd_pct
        SCOR              46             0.0001                  2                3           97.78              44.5         144.31          88.75          314.5                      15.22               39.13               60.87                 4.3478              13.0435                17.3913
        SECR         1285594             3.6130                302              374           57.82              30.0          87.34          66.00          145.0                      13.35               35.97               64.03                 3.8690               3.8713                 7.7402
         SER         1492985             4.1959                332              616           53.93          

In [5]:
# ==========================================
# CUBE 3: STATION-LEVEL PUNCTUALITY & BOTTLENECK ANALYSIS (>= 1,000 OBSERVED STOPS)
# ==========================================
print("\n=== CUBE 3: STATION-LEVEL ANALYSIS (>= 1,000 OBSERVED STOPS/YEAR) ===")

con.execute("""
CREATE OR REPLACE TEMP TABLE station_olap_cube AS
SELECT 
    station_name,
    station_full_name,
    station_zone,
    COUNT(*) AS observed_stops,
    COUNT(DISTINCT train_no) AS distinct_trains,
    ROUND(AVG(delay), 2) AS mean_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_delay_min,
    ROUND(STDDEV(delay), 2) AS std_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 2) AS iqr_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.90), 2) AS p90_delay_min,
    ROUND(AVG(is_early_or_ontime) * 100.0, 2) AS early_or_ontime_le_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    COUNT(*) FILTER (WHERE is_high_risk_only = 1) AS high_risk_stops,
    COUNT(*) FILTER (WHERE is_critical_only = 1) AS critical_stops,
    COUNT(*) FILTER (WHERE is_bottleneck_gt_2sd = 1) AS total_bottleneck_stops,
    ROUND(AVG(is_high_risk_only) * 100.0, 4) AS high_risk_2sd_3sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct
FROM vw_olap_observed
GROUP BY station_name, station_full_name, station_zone;
""")

elig_summary = q("""
SELECT 
    COUNT(*) AS total_observed_stations,
    COUNT(*) FILTER (WHERE observed_stops >= 1000) AS eligible_stations_ge_1000,
    ROUND(COUNT(*) FILTER (WHERE observed_stops >= 1000) * 100.0 / COUNT(*), 2) AS pct_stations_eligible,
    SUM(observed_stops) AS total_observed_stops,
    SUM(observed_stops) FILTER (WHERE observed_stops >= 1000) AS eligible_observed_stops,
    ROUND(SUM(observed_stops) FILTER (WHERE observed_stops >= 1000) * 100.0 / SUM(observed_stops), 2) AS pct_traffic_in_eligible_stations
FROM station_olap_cube;
""")
print("Station Eligibility Coverage (>= 1,000 Observed Stops/Year):")
print(elig_summary.to_string(index=False))

# Export full eligible station cube (>= 1,000 stops)
station_ge1000_csv = RESULT_DIR / "olap_step03_03_station_performance_ge1000.csv"
con.execute(f"""
COPY (
    SELECT * FROM station_olap_cube
    WHERE observed_stops >= 1000
    ORDER BY bottleneck_gt_2sd_pct DESC, observed_stops DESC
) TO '{station_ge1000_csv.as_posix()}' (HEADER, DELIMITER ',');
""")

top20_bottleneck_stations = q("""
SELECT 
    station_name, station_full_name, station_zone, observed_stops, distinct_trains,
    mean_delay_min, median_delay_min, on_time_le_15m_pct,
    high_risk_2sd_3sd_pct, critical_gt_3sd_pct, bottleneck_gt_2sd_pct
FROM station_olap_cube
WHERE observed_stops >= 1000
ORDER BY bottleneck_gt_2sd_pct DESC, observed_stops DESC
LIMIT 20;
""")
print("\nTop 20 High-Risk + Critical Bottleneck Stations (>= 1,000 Observed Stops):")
print(top20_bottleneck_stations.to_string(index=False))

top20_major_hub_bottlenecks = q("""
SELECT 
    station_name, station_full_name, station_zone, observed_stops, distinct_trains,
    mean_delay_min, median_delay_min, on_time_le_15m_pct,
    high_risk_2sd_3sd_pct, critical_gt_3sd_pct, bottleneck_gt_2sd_pct
FROM station_olap_cube
WHERE observed_stops >= 25000
ORDER BY bottleneck_gt_2sd_pct DESC
LIMIT 20;
""")
print("\nTop 20 Major Transit Hub Bottlenecks (>= 25,000 Observed Stops/Year):")
print(top20_major_hub_bottlenecks.to_string(index=False))

top20_punctual_stations = q("""
SELECT 
    station_name, station_full_name, station_zone, observed_stops, distinct_trains,
    mean_delay_min, median_delay_min, on_time_le_15m_pct, bottleneck_gt_2sd_pct
FROM station_olap_cube
WHERE observed_stops >= 1000
ORDER BY on_time_le_15m_pct DESC, observed_stops DESC
LIMIT 20;
""")
print("\nTop 20 Most Punctual Stations (<= 15m On-Time %, >= 1,000 Observed Stops):")
print(top20_punctual_stations.to_string(index=False))

top20_bottleneck_stations.to_csv(RESULT_DIR / "olap_step03_03_station_top20_bottlenecks_ge1000.csv", index=False)
top20_major_hub_bottlenecks.to_csv(RESULT_DIR / "olap_step03_03_station_top20_major_hubs_ge25000.csv", index=False)
top20_punctual_stations.to_csv(RESULT_DIR / "olap_step03_03_station_top20_punctual_ge1000.csv", index=False)

FINDINGS["cube3_station_analysis"] = {
    "eligibility_coverage": elig_summary.iloc[0].to_dict(),
    "top20_bottleneck_stations_ge1000": top20_bottleneck_stations.to_dict(orient="records"),
    "top20_major_hub_bottlenecks_ge25000": top20_major_hub_bottlenecks.to_dict(orient="records"),
    "top20_punctual_stations_ge1000": top20_punctual_stations.to_dict(orient="records")
}


=== CUBE 3: STATION-LEVEL ANALYSIS (>= 1,000 OBSERVED STOPS/YEAR) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Station Eligibility Coverage (>= 1,000 Observed Stops/Year):
 total_observed_stations  eligible_stations_ge_1000  pct_stations_eligible  total_observed_stops  eligible_observed_stops  pct_traffic_in_eligible_stations
                    8214                       7076                  86.15            35582318.0               34876656.0                             98.02

Top 20 High-Risk + Critical Bottleneck Stations (>= 1,000 Observed Stops):
station_name station_full_name station_zone  observed_stops  distinct_trains  mean_delay_min  median_delay_min  on_time_le_15m_pct  high_risk_2sd_3sd_pct  critical_gt_3sd_pct  bottleneck_gt_2sd_pct
        CTHR          CHITAHRA          WCR            2189                6          133.77             132.0                5.85                20.8771              11.1923                32.0694
        TKYR           TIKARIA          WCR            2189                6          129.65             127.0                4.84                20.0548  

In [6]:
# ==========================================
# CUBE 4: TEMPORAL HIERARCHY ANALYSIS (QUARTERLY, MONTHLY, WEEKDAY)
# ==========================================
print("\n=== CUBE 4A: QUARTERLY ROLL-UP (JOURNEY START DATE) ===")

quarterly_df = q("""
SELECT 
    year,
    quarter,
    COUNT(*) AS observed_stops,
    ROUND(AVG(delay), 2) AS mean_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_delay_min,
    ROUND(STDDEV(delay), 2) AS std_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.75) - QUANTILE_CONT(delay, 0.25), 2) AS iqr_delay_min,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_high_risk_only) * 100.0, 4) AS high_risk_2sd_3sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct
FROM vw_olap_observed
GROUP BY year, quarter
ORDER BY year, quarter;
""")
print(quarterly_df.to_string(index=False))
quarterly_df.to_csv(RESULT_DIR / "olap_step03_03_temporal_quarterly.csv", index=False)

print("\n=== CUBE 4B: MONTHLY DRILL-DOWN (CHRONOLOGICAL) ===")
monthly_df = q("""
SELECT 
    year,
    quarter,
    month,
    month_name,
    COUNT(*) AS observed_stops,
    ROUND(AVG(delay), 2) AS mean_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_delay_min,
    ROUND(STDDEV(delay), 2) AS std_delay_min,
    ROUND(QUANTILE_CONT(delay, 0.90), 2) AS p90_delay_min,
    ROUND(AVG(is_early_or_ontime) * 100.0, 2) AS early_or_ontime_le_0m_pct,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_high_risk_only) * 100.0, 4) AS high_risk_2sd_3sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct
FROM vw_olap_observed
GROUP BY year, quarter, month, month_name
ORDER BY year, month;
""")
print(monthly_df.to_string(index=False))
monthly_df.to_csv(RESULT_DIR / "olap_step03_03_temporal_monthly.csv", index=False)

print("\n=== CUBE 4C: DAY-OF-WEEK & WEEKEND ANALYSIS ===")
weekday_df = q("""
SELECT 
    day_of_week,
    day_name,
    is_weekend,
    COUNT(*) AS observed_stops,
    ROUND(AVG(delay), 2) AS mean_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_delay_min,
    ROUND(STDDEV(delay), 2) AS std_delay_min,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct
FROM vw_olap_observed
GROUP BY day_of_week, day_name, is_weekend
ORDER BY day_of_week;
""")
print(weekday_df.to_string(index=False))
weekday_df.to_csv(RESULT_DIR / "olap_step03_03_temporal_weekday.csv", index=False)

FINDINGS["cube4_temporal_hierarchy"] = {
    "quarterly": quarterly_df.to_dict(orient="records"),
    "monthly": monthly_df.to_dict(orient="records"),
    "weekday": weekday_df.to_dict(orient="records")
}


=== CUBE 4A: QUARTERLY ROLL-UP (JOURNEY START DATE) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 year  quarter  observed_stops  mean_delay_min  median_delay_min  std_delay_min  iqr_delay_min  on_time_le_15m_pct  delayed_gt_15m_pct  high_risk_2sd_3sd_pct  critical_gt_3sd_pct  bottleneck_gt_2sd_pct
 2025        1         4829807           37.00              15.0          73.93           36.0               50.33               49.67                 1.7052               2.1596                 3.8648
 2025        2         8909772           33.77              15.0          62.51           36.0               51.12               48.88                 1.5599               1.5536                 3.1135
 2025        3         9095450           31.90              14.0          59.41           34.0               51.89               48.11                 1.3729               1.3923                 2.7651
 2025        4         9068653           38.96              18.0          69.64           39.0               46.77               53.23                 1.9392               2.1325              

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 year  quarter  month month_name  observed_stops  mean_delay_min  median_delay_min  std_delay_min  p90_delay_min  early_or_ontime_le_0m_pct  on_time_le_15m_pct  delayed_gt_15m_pct  high_risk_2sd_3sd_pct  critical_gt_3sd_pct  bottleneck_gt_2sd_pct
 2025        1      2   February         1892863           41.23              16.0          86.44           97.0                      14.42               49.54               50.46                 1.8850               2.9568                 4.8418
 2025        1      3      March         2936944           34.27              15.0          64.46           85.0                      14.84               50.85               49.15                 1.5894               1.6458                 3.2352
 2025        2      4      April         2928999           31.43              14.0          58.75           79.0                      17.48               53.02               46.98                 1.3759               1.2976                 2.6735
 2025       

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 day_of_week  day_name  is_weekend  observed_stops  mean_delay_min  median_delay_min  std_delay_min  on_time_le_15m_pct  delayed_gt_15m_pct  bottleneck_gt_2sd_pct  critical_gt_3sd_pct
           1    Monday       False         5117702           35.96              16.0          67.88               49.60               50.40                 3.5305               1.8617
           2   Tuesday       False         5074319           35.17              16.0          65.55               49.66               50.34                 3.3234               1.7165
           3 Wednesday       False         5075133           35.18              16.0          63.91               49.65               50.35                 3.3981               1.7223
           4  Thursday       False         5104061           36.13              16.0          66.35               49.11               50.89                 3.5956               1.8436
           5    Friday       False         5131880           36.18              

In [7]:
# ==========================================
# CUBE 5: MULTIDIMENSIONAL CROSS-DRILLDOWNS (2D CUBOIDS)
# ==========================================
print("\n=== CUBE 5A: 2D CUBOID — RAILWAY ZONE x TRAIN CATEGORY (>= 1,000 STOPS) ===")

zone_train_df = q("""
SELECT 
    station_zone,
    primary_type_code,
    COUNT(*) AS observed_stops,
    COUNT(DISTINCT train_no) AS distinct_trains,
    ROUND(AVG(delay), 2) AS mean_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_delay_min,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_delayed_gt_15m) * 100.0, 2) AS delayed_gt_15m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct,
    ROUND(AVG(is_critical_only) * 100.0, 4) AS critical_gt_3sd_pct
FROM vw_olap_observed
GROUP BY station_zone, primary_type_code
HAVING COUNT(*) >= 1000
ORDER BY bottleneck_gt_2sd_pct DESC;
""")
print("Top 15 Most Delayed (Zone x Train Category) Cells:")
print(zone_train_df.head(15).to_string(index=False))
zone_train_df.to_csv(RESULT_DIR / "olap_step03_03_cuboid_zone_x_traintype.csv", index=False)

print("\n=== CUBE 5B: 2D CUBOID — MONTH x TRAIN CATEGORY ===")
month_train_df = q("""
SELECT 
    year,
    month,
    month_name,
    primary_type_code,
    COUNT(*) AS observed_stops,
    ROUND(AVG(delay), 2) AS mean_delay_min,
    ROUND(MEDIAN(delay), 2) AS median_delay_min,
    ROUND(AVG(is_on_time_15m) * 100.0, 2) AS on_time_le_15m_pct,
    ROUND(AVG(is_bottleneck_gt_2sd) * 100.0, 4) AS bottleneck_gt_2sd_pct
FROM vw_olap_observed
GROUP BY year, month, month_name, primary_type_code
ORDER BY year, month, primary_type_code;
""")
month_train_df.to_csv(RESULT_DIR / "olap_step03_03_cuboid_month_x_traintype.csv", index=False)
print(f"Exported {len(month_train_df)} (Month x Train Category) cells.")

FINDINGS["cube5_cross_dimensional_cuboids"] = {
    "top15_zone_x_traintype_by_bottleneck": zone_train_df.head(15).to_dict(orient="records"),
    "bottom10_zone_x_traintype_by_bottleneck": zone_train_df.tail(10).to_dict(orient="records")
}


=== CUBE 5A: 2D CUBOID — RAILWAY ZONE x TRAIN CATEGORY (>= 1,000 STOPS) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Top 15 Most Delayed (Zone x Train Category) Cells:
station_zone primary_type_code  observed_stops  distinct_trains  mean_delay_min  median_delay_min  on_time_le_15m_pct  delayed_gt_15m_pct  bottleneck_gt_2sd_pct  critical_gt_3sd_pct
         SER        PRM-TRAINS            3014               13          309.93             171.0               15.13               84.87                50.2654              42.1699
         NER        PRM-TRAINS           18356               24          169.90              72.0               27.15               72.85                36.2988              27.9146
         ECR        PRM-TRAINS           11025               27          186.27              72.0               26.15               73.85                35.8639              30.2494
          NR        PRM-TRAINS           11095               43          140.32              21.0               46.66               53.34                27.5710              23.3529
         NFR        PRM-TRAINS         

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Exported 104 (Month x Train Category) cells.


In [8]:
# ==========================================
# STEP 6: GENERATE OLAP VISUALIZATIONS & SAVE FINDINGS JSON
# ==========================================
print("\n=== STEP 6: GENERATING OLAP FIGURES IN results/figures/ ===")

plt.rcParams["figure.dpi"] = 150

# 1. Train Category Punctuality vs Bottleneck Rate
fig, ax1 = plt.subplots(figsize=(10, 5))
tc_plot = train_cat_df.sort_values("mean_delay_min", ascending=False)
x = range(len(tc_plot))
ax1.bar(x, tc_plot["mean_delay_min"], width=0.4, label="Mean Delay (min)", color="#2b5c8f", align="center")
ax1.set_xticks(list(x))
ax1.set_xticklabels(tc_plot["primary_type_code"], rotation=30, ha="right")
ax1.set_ylabel("Mean Delay (Minutes)", color="#2b5c8f")
ax2 = ax1.twinx()
ax2.plot(list(x), tc_plot["bottleneck_gt_2sd_pct"], color="#d95f02", marker="o", linewidth=2, label="Bottleneck >2 SD (%)")
ax2.set_ylabel("High-Risk + Critical Rate (%)", color="#d95f02")
plt.title("Train Category OLAP Roll-Up: Mean Delay vs. Bottleneck Rate (>2 SD)")
fig.tight_layout()
fig1_path = FIG_DIR / "step03_03_train_category_delay_bottleneck.png"
plt.savefig(fig1_path)
plt.close(fig)

# 2. Railway Zone Bottleneck Rate (Top 18 Zones with >= 50,000 Observed Stops)
zone_plot = zone_df[zone_df["observed_stops"] >= 50000].sort_values("bottleneck_gt_2sd_pct", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(zone_plot["station_zone"], zone_plot["high_risk_2sd_3sd_pct"], label="High-Risk (2-3 SD)", color="#fdae61")
ax.barh(zone_plot["station_zone"], zone_plot["critical_gt_3sd_pct"], left=zone_plot["high_risk_2sd_3sd_pct"], label="Critical (>3 SD)", color="#d73027")
ax.set_xlabel("Percentage of Observed Stops (%)")
ax.set_title("Railway Zone OLAP Performance: High-Risk & Critical Bottleneck Rates")
ax.legend()
fig.tight_layout()
fig2_path = FIG_DIR / "step03_03_railway_zone_bottleneck_rates.png"
plt.savefig(fig2_path)
plt.close(fig)

# 3. Monthly Temporal Trend (Mean, Median, and On-Time %)
fig, ax1 = plt.subplots(figsize=(11, 5))
m_labels = [f"{r['month_name'][:3]}\n{r['year']}" for _, r in monthly_df.iterrows()]
x_m = range(len(monthly_df))
ax1.plot(list(x_m), monthly_df["mean_delay_min"], marker="o", linewidth=2, label="Mean Delay (min)", color="#1f78b4")
ax1.plot(list(x_m), monthly_df["median_delay_min"], marker="s", linewidth=2, linestyle="--", label="Median Delay (min)", color="#33a02c")
ax1.set_xticks(list(x_m))
ax1.set_xticklabels(m_labels)
ax1.set_ylabel("Delay (Minutes)")
ax1.legend(loc="upper left")
ax2 = ax1.twinx()
ax2.plot(list(x_m), monthly_df["bottleneck_gt_2sd_pct"], marker="^", linewidth=2, color="#e31a1c", label="Bottleneck >2 SD (%)")
ax2.set_ylabel("Bottleneck Rate >2 SD (%)", color="#e31a1c")
ax2.legend(loc="upper right")
plt.title("Monthly Temporal OLAP Drill-Down: Delay & Bottleneck Trajectory (Feb 2025 – Feb 2026)")
fig.tight_layout()
fig3_path = FIG_DIR / "step03_03_monthly_temporal_trend.png"
plt.savefig(fig3_path)
plt.close(fig)

print(f"-> Saved figures:\n   {fig1_path.name}\n   {fig2_path.name}\n   {fig3_path.name}")

# Save Step 3.3 Findings JSON
findings_path = RESULT_DIR / "step03_03_OLAP_FINDINGS.json"
with open(findings_path, "w") as f:
    json.dump(FINDINGS, f, indent=4)

print(f"\n-> Saved Step 3.3 OLAP findings to {findings_path.as_posix()}")
con.close()


=== STEP 6: GENERATING OLAP FIGURES IN results/figures/ ===
-> Saved figures:
   step03_03_train_category_delay_bottleneck.png
   step03_03_railway_zone_bottleneck_rates.png
   step03_03_monthly_temporal_trend.png

-> Saved Step 3.3 OLAP findings to D:/M_Tech_CSE_AI/SEM01/DWDM/project/indian_railway_delay/results/step03_03_OLAP_FINDINGS.json
